# Method Validation
### Roadrunner propulsion reverse-engineering study · notebook 5 of 13

**Purpose.** There is no ground truth for the vehicle, so its results can never be
checked directly. What *can* be checked is the method (ADR-0012). This notebook does
two things with the purchasable-engine catalogue, whose datasheets we do have:

1. **Anchor** a pyCycle turbojet model to each datasheet and see whether a simple
   cycle can reproduce what the vendor publishes.
2. **Predict each engine blind**, from its outer diameter alone, using technology
   levels taken from the *other* engines, and compare with its datasheet.

> *If all we had of a known engine were its diameter, how well would this method
> recover its thrust, air flow and fuel consumption?*

The error found here is the error bar to quote on the vehicle study.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from roadrunner_re import load_engine_catalogue, reports
from roadrunner_re import technology as tech
from roadrunner_re import validation as val
from roadrunner_re.cycle import load_cycle_settings
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots.validation import blind_predictions

paths = nb_setup()
engines = load_engine_catalogue()
metrics = tech.catalogue_metrics(engines, tech.load_fuel_density().value, tech.load_gravity())

---
## 1 · The cycle model and what it has to assume

A single-spool turbojet in pyCycle (ADR-0015): inlet, centrifugal compressor,
combustor, single-stage turbine, convergent nozzle; stations numbered per ARP755.
Thermodynamics are pyCycle's tabular Jet-A data. Vendors publish thrust, air flow,
fuel flow and sometimes pressure ratio and exhaust temperature. Everything else is
assumed in `config/cycle.yaml` — **do not hard-code numbers in this notebook.**

In [ ]:
settings = load_cycle_settings()
for name, value in settings.nominal_losses().items():
    u = getattr(settings, name)
    print(f"{name:30s}{value:7.3f}   bounds {u.low:.2f} to {u.high:.2f}")
u = settings.pressure_ratio_when_unpublished
print(f"{'pressure ratio if unpublished':30s}{u.value:7.3f}   bounds {u.low:.2f} to {u.high:.2f}")

---
## 2 · Anchoring the model to each datasheet

### Theory

At sea-level static conditions, with air flow $\dot m_a$ and pressure ratio
$\pi_c$ taken from the datasheet and the component losses assumed, the net thrust
depends on one remaining unknown, the turbine inlet temperature $T_{t4}$. The solver
finds the $T_{t4}$ that gives the published thrust.

The model burns its fuel completely, so its fuel flow is an ideal one. The ratio

$$\eta_b = \frac{\dot m_{f,\,\text{model}}}{\dot m_{f,\,\text{published}}}$$

is reported as a **lumped combustion efficiency**: it absorbs real combustion
inefficiency, the oil these engines burn with their fuel, and any error in the assumed
losses.

The published exhaust gas temperature (EGT) is **not** used in the match. Comparing
it with the model's turbine exit temperature $T_{t5}$ is therefore an independent
check.

In [ ]:
anchors = val.anchor_catalogue(engines, metrics, settings)
print(reports.anchor_table(anchors))
print("\n* pressure ratio not published; catalogue value assumed")

Engines with no published air mass flow cannot be anchored and are absent from the
table. Two things stand out:

- **Combustion efficiency falls with engine size.** The largest anchored engines sit
  near 0.94 to 0.99; the smallest need 0.75 to 0.83 to account for their published
  fuel flow.
- **The model's turbine exit temperature does not match the published EGT**, and
  not even in a consistent direction. A one-dimensional cycle cannot reconcile a
  vendor's thrust, air flow, fuel flow and EGT all at once. A single probe in a small
  exhaust is not a mass-averaged total temperature, so the published EGT is weak
  evidence; but the disagreement is recorded as a standing finding rather than
  explained away.

---
## 3 · Predicting each engine blind

For each engine in turn, only the **outer diameter** is taken from its own datasheet,
as imagery would give for the vehicle. From the *other* anchored engines come three
technology ranges: air flow per frontal area, turbine inlet temperature, and
combustion efficiency. Then:

$$\dot m_a = \left(\frac{\dot m_a}{A}\right) \frac{\pi D^2}{4}, \qquad
F = F_s(T_{t4}, \pi_c)\,\dot m_a, \qquad
\mathrm{SFC} = \frac{\mathrm{SFC}_\text{ideal}(T_{t4}, \pi_c)}{\eta_b}$$

with $F_s$ and the ideal SFC from pyCycle at the nominal point and at the corners of
the technology ranges. Variants of one core are left out together. A prediction
**passes** if the published value lies inside the predicted interval and the interval
is no wider than the limit set beforehand in `config/validation.yaml`.

In [ ]:
limits = val.load_validation_limits()
predictions = val.leave_one_out(engines, metrics, anchors, settings, limits)

print(reports.blind_prediction_table(predictions, "thrust_N", unit="[N]"), end="\n\n")
print(reports.blind_prediction_table(predictions, "air_mass_flow_kg_s", unit="[kg/s]"), end="\n\n")
print(reports.blind_prediction_table(predictions, "sfc_kg_N_s", tech.SECONDS_PER_HOUR, "[kg/(N h)]"))

In [ ]:
fig = blind_predictions(predictions, paths.figures / "method_validation.png")

In [ ]:
for quantity, count in val.pass_counts(predictions).items():
    print(f"{quantity:22s} {count['passed']} of {count['of']} pass   (width limit {limits[quantity]:.0%} of nominal)")

---
## 4 · Handoff

In [ ]:
handoff = val.write_validation_handoff(val.validation_handoff(anchors, predictions), paths.out / "validation.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **From diameter alone, the method recovers thrust to within about ±13 % at the
  nominal**, and the published thrust lies inside the predicted interval for every
  engine tested. The interval itself is wide, roughly ±25 % of nominal, because it
  multiplies the uncertainty in air flow per frontal area by that in specific thrust.
- **SFC is recovered within about ±18 %**, with the published value inside the
  interval wherever one is published.
- **Air flow fails for three of eight predictions** (two of them variants of one
  core): the engines with the highest and the lowest air flow per frontal area fall
  outside the range of the others, by about 16 %. That is what leave-one-out is for:
  it shows the catalogue's own range does not always contain the next engine. Thrust
  still passes for those engines because specific thrust compensates.
- **Standing finding:** the model's turbine exit temperature differs from published
  EGT by up to about 130 K in either direction.
- **Standing finding:** the anchored turbine inlet temperatures depend on the assumed
  compressor and turbine efficiencies, which no vendor publishes. They are plausible
  for uncooled small turbines but they are not measurements.
- These errors are the ones to quote next to any thrust or fuel-consumption figure
  derived for the vehicle from a measured nacelle diameter.